# E-Commerce Delivery Performance Analysis with SQL

## Business Question

What percentage of delivered orders arrived after the estimated delivery date,
and which Brazilian states had the highest late delivery rates?

## Key Findings

- **Overall:** 6,534 of 96,470 eligible orders arrived late (6.77%).
- **Highest late delivery rate:** Alagoas had a late delivery rate of 21.41%
  (85 of 397 orders).
- **Most late deliveries:** São Paulo had 1,820 late deliveries, with a
  late delivery rate of 4.49%.
- **Investigation priority:** Rio de Janeiro had 1,495 late deliveries
  and a late delivery rate of 12.11%, above the overall rate of 6.77%.

## Dataset

I used the [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
The orders table provides order status and delivery dates. I joined it
to the customers table on `customer_id` to identify each customer's state.

## How I Measure Late Delivery

An order counts as late if it arrived after the estimated delivery day.
I compare calendar dates, so an order that arrived on the estimated day
counts as on time. The late delivery rate uses delivered orders with
both dates available. Eight delivered orders were excluded because at
least one of the two dates was missing.

In [2]:
import sqlite3
import pandas as pd

orders = pd.read_csv("/content/olist_orders_dataset.csv")
db = sqlite3.connect(":memory:")
orders.to_sql("orders", db, index=False, if_exists="replace")

99441

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
pd.read_sql_query("""
SELECT order_id, order_status
FROM orders
LIMIT 5
""", db)

,order_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered


## 1. Count delivered orders

I start by counting orders marked as `delivered`. This gives me a baseline
before I check how many of those orders have both dates needed to measure delays.

In [6]:
pd.read_sql_query("""
SELECT COUNT(*) AS delivered_orders
FROM orders
WHERE order_status = 'delivered'
""", db)

,delivered_orders
0,96478


## 2. Check delivery date completeness

I check how many delivered orders have both the actual and estimated
delivery dates. Orders with both dates will form the denominator for
the late delivery rate.

In [9]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS delivered_orders,
    COUNT(
        CASE
            WHEN order_delivered_customer_date IS NULL
              OR order_estimated_delivery_date IS NULL
            THEN 1
        END
    ) AS missing_either_date,
    COUNT(
        CASE
            WHEN order_delivered_customer_date IS NOT NULL
             AND order_estimated_delivery_date IS NOT NULL
            THEN 1
        END
    ) AS orders_with_both_dates
FROM orders
WHERE order_status = 'delivered'
""", db)

,delivered_orders,missing_either_date,orders_with_both_dates
0,96478,8,96470


## 3. Calculate the late delivery rate

Of the 96,478 delivered orders, 8 were missing at least one delivery date.
I compare actual and estimated calendar dates for the remaining 96,470 orders.
An order counts as late if it arrived after the estimated delivery day.

In [10]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS eligible_orders,
    SUM(
        CASE
            WHEN DATE(order_delivered_customer_date) >
                 DATE(order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * AVG(
            CASE
                WHEN DATE(order_delivered_customer_date) >
                     DATE(order_estimated_delivery_date)
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS late_delivery_pct
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
""", db)

,eligible_orders,late_orders,late_delivery_pct
0,96470,6534,6.77


**Finding:** Of the 96,470 delivered orders with both delivery dates available,
6,534 (6.77%) arrived after the estimated delivery date.

In [12]:
customers = pd.read_csv("/content/olist_customers_dataset.csv")
customers.to_sql("customers", db, index=False, if_exists="replace")

pd.read_sql_query("""
SELECT customer_id, customer_state
FROM customers
LIMIT 5
""", db)

,customer_id,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,SP
1,18955e83d337fd6b2def6b18a428ac77,SP
2,4e7b3e00288586ebd08712fdd0374a03,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,SP


## 4. Compare late delivery rates by customer state

I join the orders and customers tables on `customer_id` to identify the
customer's state for each order. For each state, I report the number of
delivered orders with both dates available, the number of late orders,
and the late delivery rate.

In [13]:
state_results = pd.read_sql_query("""
SELECT
    c.customer_state,
    COUNT(*) AS eligible_orders,
    SUM(CASE
        WHEN DATE(o.order_delivered_customer_date) >
             DATE(o.order_estimated_delivery_date)
        THEN 1 ELSE 0
    END) AS late_orders,
    ROUND(100.0 * AVG(CASE
        WHEN DATE(o.order_delivered_customer_date) >
             DATE(o.order_estimated_delivery_date)
        THEN 1 ELSE 0
    END), 2) AS late_delivery_pct
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_delivery_pct DESC, eligible_orders DESC
""", db)

state_results.head(10)

,customer_state,eligible_orders,late_orders,late_delivery_pct
0,AL,397,85,21.41
1,MA,717,125,17.43
2,SE,335,51,15.22
3,PI,476,66,13.87
4,CE,1279,176,13.76
5,RR,41,5,12.20
6,BA,3256,396,12.16
7,RJ,12350,1495,12.11
8,PA,946,106,11.21
9,ES,1995,214,10.73


**Finding:** Alagoas (AL) had the highest late delivery rate at 21.41%
(85 of 397 orders). Rio de Janeiro (RJ) recorded 1,495 late deliveries,
accounting for 22.9% of all late deliveries in the dataset. I would
investigate Rio de Janeiro first to understand where delays affected
a large number of orders.

In [14]:
state_results.sort_values("late_orders", ascending=False).head(5)

,customer_state,eligible_orders,late_orders,late_delivery_pct
21,SP,40494,1820,4.49
7,RJ,12350,1495,12.11
20,MG,11354,519,4.57
6,BA,3256,396,12.16
17,RS,5344,325,6.08


### Key findings by state

- Alagoas had the highest late delivery rate at 21.41% (85 of 397 orders).
- São Paulo had the most late deliveries (1,820), with a late delivery
  rate of 4.49%.
- Rio de Janeiro had the second-most late deliveries (1,495) and a late
  delivery rate of 12.11%, above the overall rate of 6.77%.

São Paulo and Rio de Janeiro accounted for 50.7% of all late deliveries.
I would start the investigation in Rio de Janeiro because it combines
a high number of late deliveries with an above-average late delivery rate.

In [15]:
state_totals = state_results[["eligible_orders", "late_orders"]].sum()

print("Eligible orders:", state_totals["eligible_orders"])
print("Late orders:", state_totals["late_orders"])

Eligible orders: 96470
Late orders: 6534


## Recommendation

I recommend investigating Rio de Janeiro first because it had both a high
number of late deliveries and a late delivery rate above the overall rate.
As a follow-up, I would compare late delivery rates and order volumes by
month and seller to see where delays were concentrated.